# Opening and closing the door

We will look at messages sent to the workshop plate reader and find out which bytes open and close its door.

Open **[captures/door.txt](captures/door.txt)** in a text editor next to this notebook. Keep it open. We will read the file ourselves and copy one line at a time into Python.

Run a code cell with **Shift+Enter**. The transcript exercises need only Python. For the hands-on section, connect your workshop Pico and have its COM/serial port ready. We will use pySerial to communicate with it; Wireshark is not needed.

The text file shows messages exchanged between the computer and the plate reader.

The Pico's onboard LED is on when the simulated door is open and off when it is closed.


## 1. Find the Open message and inspect it in Python

In `door.txt`, find `# Action: Click Open`. The two lines directly below it are:

```text
> A7 03 06 00 00 00 00 FF
< A7 83 06 00 00 00 00 FF
```

- `>` means the computer **sends** a message to the reader.
- `<` means the reader **replies** to the computer.

Each pair of hexadecimal digits is one byte. The `>` and `<` labels are not part of the message.

**Your turn:** count the bytes in the outgoing message, then run the cell below to check. The Open message is filled in for you. `bytes.fromhex()` converts the hex into bytes; `.hex(" ")` displays them again.


In [ ]:
open_message = bytes.fromhex("A7 03 06 00 00 00 00 FF")
print("Number of bytes:", len(open_message))
print("Message:", open_message.hex(" "))


## 2. Replicate the exchange on your Pico

It is always a good idea to see if we can replicate an exchange and get the same response. Before investigating Close, let’s send the Open message to your Pico and compare its reply with the text file. It must already be running the workshop firmware. The Pico simulates the reader's door; there is no physical door to watch, so we will ask it for position information.

Use a Python kernel running on the computer the Pico is plugged into. Close the desktop client, Thonny's device connection, and any serial monitor first: this notebook needs the serial port to itself.

We use **pySerial** to send and receive bytes. If `import serial` below fails because the module is missing, uncomment and run the installation cell, then try again.

In [ ]:
# %pip install pyserial

Enter the COM/serial port you already know in place of `YOUR_PORT`. For example, Windows ports look like `COM3`; macOS and Linux use paths such as `/dev/cu.usbmodem...` or `/dev/ttyACM0`.

`serial.Serial(...)` opens the connection. The baud rate is the workshop connection setting, not something we inferred from the transcript. `timeout=1` allows each read to wait up to one second. The short pause lets startup text settle; `reset_input_buffer()` discards any old input before our first request.

Run this connection cell once. To reconnect, run the disconnect cell at the end first.

In [ ]:
import serial
import time

PORT = "YOUR_PORT"  # Replace with your Pico's COM/serial port.
pico = serial.Serial(PORT, baudrate=115200, timeout=1, write_timeout=1)
time.sleep(1)
pico.reset_input_buffer()
print("Connected to", pico.port)

### Send Open and compare the reply

We observed an eight-byte Open reply in the text file, so we will ask for eight bytes here.

`write()` sends the actual bytes. `read(8)` waits for eight bytes or the timeout. It can return fewer bytes if the complete reply does not arrive in time; the length check makes us stop in that case.

**Predict:** what reply do you expect? Run the cell, then compare the printed reply with the `<` line under `Click Open` in `door.txt`.

Always read one reply before sending the next request.

In [ ]:
open_request = bytes.fromhex("A7 03 06 00 00 00 00 FF")
pico.write(open_request)
reply = pico.read(8)
print("Open reply:", reply.hex(" "))
assert len(reply) == 8, "Incomplete reply: stop and check the connection."
expected_reply = bytes.fromhex("A7 83 06 00 00 00 00 FF")
print("Same reply as the text file?", reply == expected_reply)


If everything is right, the Pico's onboard LED is now on.

## 3. Turn the Open exchange into a function

We have sent the Open command and checked its reply. Now put that working exchange inside `open_door()`, so other code can ask to open the door without knowing the command bytes or how to receive the reply.

**Your turn:** copy the confirmed Open bytes from section 2 into the function below. The function creates its own request each time it is called and uses our existing `pico` connection. It does not need the earlier `open_message` or `open_request` variables.

The write and read belong together inside the function: each call sends one request and receives its reply. `return` gives those reply bytes back to the caller. Running this definition creates the function; calling `open_door()` performs the exchange.


In [ ]:
def open_door():
    request = bytes.fromhex("")  # Copy the confirmed Open command here.
    assert len(request) == 8, "Copy all eight request bytes first."
    pico.write(request)
    reply = pico.read(8)
    assert len(reply) == 8, "Incomplete Open reply."
    return reply


### Call your function

After filling in the bytes and running the definition, run the cell below. Compare its reply with the Open reply in `door.txt`.


In [ ]:
open_reply = open_door()
print("Open reply:", open_reply.hex(" "))


The caller now uses `open_door()` to describe the action. The function hides the command bytes and the write/read steps: this is the abstraction a driver provides. We will check the door's reported position later in this notebook.


## 4. Find the Close message

Back in `door.txt`, find `# Action: Click Close`.

Look at the `>` line directly below it. Compare it with the `>` line under `Click Open`.

**Your turn:** which byte changes? Write down the value for Open and the value for Close.

Copy the bytes from the Close line into the empty quotation marks below, then run the cell.


In [ ]:
close_message = bytes.fromhex("")  # Copy the Close request here, without >.
print("Number of bytes:", len(close_message))
print("Close message:", close_message.hex(" "))
print("Same as Open?", close_message == open_message)


### Create `close_door()`

Use the same structure as `open_door()`: build the Close request inside the function, send it, read its eight-byte reply, check the length, and return the reply. Then call it and compare the response with the Close exchange in the transcript.


In [ ]:
def close_door():
    # Build the Close request and complete the write/read exchange.
    # Check the reply length, then return the reply.
    pass


In [ ]:
# Run after implementing close_door():
# close_reply = close_door()
# print("Close reply:", close_reply.hex(" "))


## 5. Identify the command byte

Python starts counting at zero:

```text
Index:       0     1     2     3     4     5     6     7
Open:       A7    03    06    00    00    00    00    FF
```

The third byte has index `2`. Use `[2]` to inspect it.


In [ ]:
print("Open command candidate:", open_message[2])
print("First two bytes:", open_message[:2].hex(" "))
print("Bytes after the candidate:", open_message[3:].hex(" "))


**Your turn:** after filling in `close_message`, remove the `#` characters below and run the cell. Compare the same three parts of both messages. Is the command candidate the only part that changes?


In [ ]:
# print("Close command candidate:", close_message[2])
# print("Same first two bytes?", open_message[:2] == close_message[:2])
# print("Same bytes after the candidate?", open_message[3:] == close_message[3:])


Look again at the outgoing lines immediately below `Click Open` and `Click Close` in the text file.

**Your turn:** based on the byte that changes between these two actions, which index would you tentatively call the command byte? Which values would you associate with Open and Close? Explain the evidence. What would repeating the clicks help you check?


## 6. Identify the likely status request

In [door.txt](captures/door.txt), look at the traffic while the application is idle, before any button clicks. Which request keeps recurring? Identify its command byte and compare it with Open and Close.

The application is probably polling the reader for status so it can keep its display up to date. Idle traffic gives us that hypothesis, but does not yet tell us what the reply contains.

To investigate, we can send this request ourselves with the door open, close the door, then send the exact same request again. Any difference in the replies may reveal a position field.


## 7. Try the suspected status request

Send the request identified in the idle traffic and inspect the returned bytes. The captured replies contain 12 bytes, so use that length for this exchange.


In [ ]:
status_request = bytes.fromhex("")  # Request from the idle traffic.
assert len(status_request) == 8, "Copy all eight request bytes first."
pico.write(status_request)
status_reply = pico.read(12)
print("Reply:", status_reply.hex(" "))
print("Reply length:", len(status_reply))
assert len(status_reply) == 12, "Incomplete reply: stop and check the connection."


### Wrap the exchange in `request_state()`

Put this exchange in a function, as you did for Open and Close. Include the request bytes inside `request_state()`, read and check the 12-byte reply, and return it.

For now, the function returns the raw bytes. We will investigate their meaning by changing the door position.


In [ ]:
def request_state():
    # Build the status request and complete the write/read exchange.
    # Check the reply length, then return the reply bytes.
    pass


## 8. Test the query with Open and Close

Use `open_door()` and `close_door()` to change the position, calling `request_state()` after each. Both saved replies now come from the same query function.

If the reply contains position information, some part of it should change. Other fields may stay the same; their meanings are still unknown.


In [ ]:
open_door()
state_after_open = request_state()
print("State while open:", state_after_open.hex(" "))


In [ ]:
close_door()
state_after_close = request_state()
print("State while closed:", state_after_close.hex(" "))

print("Index / open / closed")
for index in range(len(state_after_open)):
    print(index, state_after_open[index], state_after_close[index])


Which reply byte changed, and which value corresponds to each door position? Repeat Open → query → Close → query to see whether the pattern holds. This gives us evidence for what the suspected status request returns.


### Decode the position into a Python value

The comparison shows `01` at index `9` while the door is open and `00` while it is closed. That supports interpreting this byte as position.

Indexing the reply gives a Python integer. Comparing it with `1` turns the observed position into a Boolean that other code can use.


In [ ]:
door_was_open = state_after_open[9] == 1
door_is_open = state_after_close[9] == 1
print("Open before closing:", door_was_open)
print("Open after closing:", door_is_open)


## 9. Disconnect

Run this when finished, or before opening the port again. Closing the serial connection releases it for other applications; it does not send a door command.

In [ ]:
pico.close()
print("Serial connection closed.")

### If you get stuck

- **Port not found or busy:** check your port name and close other applications using the Pico.
- **Empty or short reply:** stop sending commands. Run the disconnect cell, check that the Pico is running the workshop firmware, then reconnect and restart at Open. The checks above deliberately stop before interpreting incomplete bytes.
- **Unexpected bytes:** compare your request with the text file, including every zero. Reconnect before trying again so an unread reply cannot be mistaken for the next one.

For this first experiment, we use the reply lengths we observed for these specific messages. This is not yet a general reader for every command.

Reference for the connection methods: [pySerial API](https://pyserial.readthedocs.io/en/latest/pyserial_api.html).